In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Qiskit Runtime local testing mode

<span id="test-locally"></span>


<details>
<summary><b>เวอร์ชันของแพ็กเกจ</b></summary>

โค้ดในหน้านี้พัฒนาโดยใช้ requirements ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
ใช้ local testing mode (มีให้ใช้กับ `qiskit-ibm-runtime` v0.22.0 หรือใหม่กว่า) เพื่อทดสอบโปรแกรมก่อนที่จะปรับแต่งและส่งไปยังฮาร์ดแวร์ควอนตัมจริง หลังจากใช้ local testing mode เพื่อยืนยันโปรแกรมแล้ว สิ่งที่ต้องเปลี่ยนคือชื่อ Backend เท่านั้นเพื่อรันบน QPU

ในการใช้ local testing mode ให้ระบุ fake backends จาก ``qiskit_ibm_runtime.fake_provider`` หรือระบุ Qiskit Aer Backend เมื่อสร้าง Qiskit Runtime primitive หรือ Session

- **Fake backends**: [fake backends](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider) ใน ``qiskit_ibm_runtime.fake_provider`` จำลองพฤติกรรมของ IBM&reg; QPUs โดยใช้ QPU snapshots ซึ่ง QPU snapshots มีข้อมูลสำคัญเกี่ยวกับ QPU เช่น coupling map, basis gates, และคุณสมบัติของ Qubit ที่มีประโยชน์สำหรับการทดสอบ Transpiler และการ simulate QPU แบบมี noise โดย noise model จาก snapshot จะถูกใช้งานโดยอัตโนมัติระหว่างการ simulation
- **Aer simulator**: Simulators จาก [Qiskit Aer](/guides/simulate-with-qiskit-aer) ให้การ simulation ประสิทธิภาพสูงที่รองรับ circuits ขนาดใหญ่และ [custom noise models](/guides/build-noise-models) มีตัวเลือก simulation method หลายแบบเมื่อใช้ `AerSimulator` ใน local testing mode ดู[ตัวอย่าง Clifford simulation mode](#clifford-sim) ที่แสดงวิธี simulate Clifford circuits ที่มี qubits จำนวนมากอย่างมีประสิทธิภาพ

    <details>
    <summary>**รายการ simulation methods ที่ใช้ได้จาก Qiskit Aer**</summary>

    ดูเอกสาร [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) สำหรับข้อมูลเพิ่มเติม

    * ``"automatic"``: simulation method เริ่มต้น เลือก simulation method โดยอัตโนมัติตาม Circuit และ noise model

    * ``"statevector"``: การ simulation statevector แบบ dense ที่สามารถสุ่มผลการวัดจาก circuits *อุดมคติ* ที่มีการวัดทั้งหมดที่ท้าย Circuit สำหรับการ simulation แบบมี noise แต่ละ shot จะสุ่ม noisy circuit จาก noise model

    * ``"density_matrix"``: การ simulation density matrix ที่สามารถสุ่มผลการวัดจาก circuits *แบบมี noise* ที่มีการวัดทั้งหมดที่ท้าย Circuit

    * ``"stabilizer"``: Clifford stabilizer state simulator ที่มีประสิทธิภาพซึ่ง simulate Clifford circuits แบบมี noise ได้หากข้อผิดพลาดทั้งหมดใน noise model เป็น Clifford errors

    * ``"extended_stabilizer"``: approximate simulator สำหรับ Clifford + T circuits ที่อิงจากการแยก state เป็น ranked-stabilizer state จำนวน terms จะเพิ่มขึ้นตามจำนวน non-Clifford (T) gates

    * ``"matrix_product_state"``: tensor-network statevector simulator ที่ใช้ Matrix Product State (MPS) representation สำหรับ state ทำได้โดยมีหรือไม่มีการตัด MPS bond dimensions ขึ้นอยู่กับตัวเลือก simulator พฤติกรรมเริ่มต้นคือไม่ตัด

    * ``"unitary"``: การ simulation unitary matrix แบบ dense ของ Circuit อุดมคติ จำลอง unitary matrix ของ Circuit เอง แทนที่จะเป็นการวิวัฒนาการของ quantum state เริ่มต้น method นี้ simulate ได้เฉพาะ gates เท่านั้น ไม่รองรับการวัด reset หรือ noise

    * ``"superop"``: การ simulation superoperator matrix แบบ dense ของ Circuit อุดมคติหรือแบบมี noise จำลอง superoperator matrix ของ Circuit เอง แทนที่จะเป็นการวิวัฒนาการของ quantum state เริ่มต้น method นี้ simulate gates และ resets แบบอุดมคติและแบบมี noise ได้ แต่ไม่รองรับการวัด

    * ``"tensor_network"``: การ simulation อิงจาก tensor-network ที่รองรับทั้ง statevector และ density matrix ปัจจุบันใช้ได้เฉพาะกับ GPU และเร่งความเร็วด้วย cuQuantum `cuTensorNet` APIs
    </details>

> **Note:** - สามารถระบุตัวเลือก Qiskit Runtime ทั้งหมดใน local testing mode ได้ อย่างไรก็ตาม ตัวเลือกทั้งหมดยกเว้น shots จะถูกละเว้นเมื่อรันบน local simulator
> - แนะนำให้ติดตั้ง Qiskit Aer ก่อนใช้ fake backends หรือ Aer simulators โดยรัน `pip install qiskit-aer` fake backends ใช้ Aer simulators เบื้องหลังหากมีเพื่อใช้ประโยชน์จากประสิทธิภาพ


## ตัวอย่าง Fake backends

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## ตัวอย่าง AerSimulator
ตัวอย่างกับ sessions โดยไม่มี noise:

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

ในการ simulate แบบมี noise ให้ระบุ QPU (ฮาร์ดแวร์ควอนตัม) และส่งไปยัง Aer Aer จะสร้าง noise model จากข้อมูล calibration ของ QPU นั้น และสร้าง Aer Backend พร้อม model นั้น หากต้องการ สามารถ [สร้าง noise model](/guides/build-noise-models) เองได้

> **Caution:** QPU อาจได้รับผลกระทบจาก noise หลายประเภท Qiskit Aer noise model ที่ใช้ที่นี่จำลองได้เฉพาะบางส่วนเท่านั้น ดังนั้นจึงมีแนวโน้มว่าจะรุนแรงน้อยกว่า noise บน QPU จริง
> 
> สำหรับรายละเอียดว่า errors ใดบ้างที่รวมอยู่เมื่อ initialize noise model จาก QPU ดู Aer [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend) API reference

ตัวอย่างแบบมี noise:

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### Clifford simulation
เนื่องจาก Clifford circuits สามารถ simulate ได้อย่างมีประสิทธิภาพพร้อมผลลัพธ์ที่ยืนยันได้ Clifford simulation จึงเป็นเครื่องมือที่มีประโยชน์มาก สำหรับตัวอย่างเชิงลึก ดู [Efficient simulation of stabilizer circuits with Qiskit Aer primitives](/guides/simulate-stabilizer-circuits)

ตัวอย่าง:

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## ขั้นตอนถัดไป
> **Tip:** - ดูตัวอย่าง primitives โดยละเอียดที่ [primitives examples.](/guides/primitives-examples)
>     - อ่าน [Migrate to V2 primitives](/guides/v2-primitives)
>     - ฝึกใช้ primitives ผ่าน [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) ใน IBM Quantum Learning
>     - เรียนรู้วิธี transpile ในเครื่องในส่วน [Transpile](/guides/transpile)
>     - ลองทำ [Compare transpiler settings](/guides/circuit-transpilation-settings#compare-transpiler-settings) tutorial